In [4]:
# Python
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [5]:
# 1. Load Data
data = pd.read_csv('train.csv')
X = data.drop('label', axis=1)
y = data['label']

# 2. Normalize the pixel values (CRITICAL for preventing NaN in Softmax)
X = X / 255.0

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=69)

# 4. Reshape for the Neural Network (Features x Examples)
X_train = np.array(X_train).T
y_train = np.array(y_train)

X_test = np.array(X_test).T
y_test = np.array(y_test)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (784, 39900)
y_train shape: (39900,)


In [6]:
def init_params(): 
    # Use rand() - 0.5 to keep initial weights small and centered around 0
    w1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    w2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return w1, b1, w2, b2

def Relu(z):
    return np.maximum(0, z)

def softmax(z):
    # Subtracting the max value prevents np.exp() from overflowing
    exp = np.exp(z)
    sum_exp = np.sum(exp, axis=0, keepdims=True)
    return exp / sum_exp

def forward_prop(w1, b1, w2, b2, X):
    z1 = np.dot(w1, X) + b1
    a1 = Relu(z1)
    z2 = np.dot(w2, a1) + b2
    a2 = softmax(z2)
    return z1, a1, z2, a2

def one_hot(y):
    # Hardcoded to 10 classes to avoid bugs if a batch misses a digit
    one_hot_y = np.zeros((y.size, 10))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y.T

def derivative_Relu(z):
    return z > 0

def backward_prop(z1, a1, z2, a2, w2, X, y):
    m = y.size
    one_hot_y = one_hot(y)
    
    dz2 = a2 - one_hot_y
    dw2 = (1/m) * np.dot(dz2, a1.T)
    # Added keepdims=True to maintain (10, 1) shape
    db2 = (1/m) * np.sum(dz2, axis=1, keepdims=True) 
    
    dz1 = np.dot(w2.T, dz2) * derivative_Relu(z1)
    dw1 = (1/m) * np.dot(dz1, X.T)
    # Added keepdims=True to maintain (10, 1) shape
    db1 = (1/m) * np.sum(dz1, axis=1, keepdims=True)
    
    return dw1, db1, dw2, db2

def update_params(w1, b1, w2, b2, dw1, db1, dw2, db2, learning_rate):
    w1 = w1 - learning_rate * dw1
    b1 = b1 - learning_rate * db1
    w2 = w2 - learning_rate * dw2
    b2 = b2 - learning_rate * db2
    return w1, b1, w2, b2

def predict(a2):
    return np.argmax(a2, 0)

def get_accuracy(predictions, y):
    return np.sum(predictions == y) / y.size

def gradient_descent(X, y, learning_rate=0.1, epochs=100):
    w1, b1, w2, b2 = init_params()
    for i in range(epochs):
        z1, a1, z2, a2 = forward_prop(w1, b1, w2, b2, X)
        dw1, db1, dw2, db2 = backward_prop(z1, a1, z2, a2, w2, X, y)
        w1, b1, w2, b2 = update_params(w1, b1, w2, b2, dw1, db1, dw2, db2, learning_rate)
        
        if i % 500 == 0:
            print(f"Iteration: {i}")
            print(f"Accuracy: {get_accuracy(predict(a2), y):.4f}")
            
    return w1, b1, w2, b2

In [7]:
w1, b1, w2, b2 = gradient_descent(X_train, y_train, learning_rate=0.1, epochs=10001)

Iteration: 0
Accuracy: 0.1146
Iteration: 500
Accuracy: 0.8587
Iteration: 1000
Accuracy: 0.8853
Iteration: 1500
Accuracy: 0.8964
Iteration: 2000
Accuracy: 0.9039
Iteration: 2500
Accuracy: 0.9094
Iteration: 3000
Accuracy: 0.9134


KeyboardInterrupt: 

In [ ]:
z1_train,  a1_train, z2_train, a2_train = forward_prop(w1, b1, w2, b2, X_train)
train_predictions = predict(a2_train)
train_accuracy = get_accuracy(train_predictions, y_train)
print(f"Final Training Accuracy: {100*train_accuracy:.2f} %")

In [ ]:
z1_test, a1_test, z2_test, a2_test = forward_prop(w1, b1, w2, b2, X_test)
testing_predictions = predict(a2_test)
test_accuracy = get_accuracy(testing_predictions, y_test)
print(f"Final Testing Accuracy: {100 * test_accuracy:.2f} %")